# ksnctf Problem 11 - Riddle

## 問題概要
複数の謎かけに答え、最終的にFLAGを入力する問題。
4問目「What is the flag?」の答えをリバースエンジニアリングで求める。

## 使用ツール
- x32dbg（動的解析）
- Rust（復号スクリプト）

## 解析手順

### 1. 暗号化方式の特定
x32dbgで`sub_6B17D0`を解析した結果、**RC4暗号**が使われていることが判明。
- Sボックス: `0x6B5578`（256バイト）
- XOR命令: `xor byte ptr ds:[eax+esi], cl`

### 2. 既知平文攻撃
problem1 Test Problemの答え`FLAG_SRORGLnTh2Q5fTwu`を仮入力し、以下の値をx32dbgで取得：
- 暗号化済み入力（ESI）
- 正解暗号文（ECX）

### 3. キーストリーム逆算
XORの対称性を利用：
```
キーストリーム = 既知平文 XOR 暗号化済み入力
正解          = 正解暗号文 XOR キーストリーム
```

In [4]:
use std::fs;
{
    // 入力文字列（既知平文）
    let plaintext_input: &[u8] = b"FLAG_SRORGLnTh2Q5fTwu";

    // 暗号化済み入力（ESIが指すメモリの内容）
    let encrypted_input: &[u8] = &[
        0xBF, 0xFF, 0x1B, 0x0D, 0x47, 0xB9, 0x0F, 0x79,
        0xCB, 0xA5, 0x4A, 0x51, 0xF5, 0x48, 0x75, 0x4E,
        0xC0, 0xE0, 0xF7, 0xDB, 0xB2,
    ];

    // 正解暗号文（ECXが指すメモリの内容）
    let correct_encrypted: &[u8] = &[
        0xBF, 0xFF, 0x1B, 0x0D, 0x47, 0xA7, 0x18, 0x4F,
        0xCB, 0xD6, 0x5C, 0x59, 0x95, 0x61, 0x73, 0x57,
        0xB3, 0xD1, 0x94, 0x9F, 0xAC,
    ];

    // Step 1: キーストリームを逆算
    // キーストリーム = 既知平文 XOR 暗号化済み入力
    let keystream: Vec<u8> = plaintext_input
        .iter()
        .zip(encrypted_input.iter())
        .map(|(p, e)| p ^ e)
        .collect();

    println!("キーストリーム:");
    for (i, k) in keystream.iter().enumerate() {
        print!("{:02X} ", k);
        if (i + 1) % 8 == 0 { println!(); }
    }
    println!();

    // Step 2: 正解文字列を復元
    // 正解 = 正解暗号文 XOR キーストリーム
    let answer: Vec<u8> = correct_encrypted
        .iter()
        .zip(keystream.iter())
        .map(|(c, k)| c ^ k)
        .collect();

    fs::write("flag.txt", &answer).unwrap();
}

キーストリーム:
F9 B3 5A 4A 18 EA 5D 36 
99 E2 06 3F A1 20 47 1F 
F5 86 A3 AC C7 


()

## 考察

### 暗号方式
RC4は**ストリーム暗号**であり、同じキーとSボックスの状態であれば
暗号化と復号が同一の処理になる。

### 既知平文攻撃の有効性
FLAGの形式（`FLAG_`プレフィックス）が既知であるため、
RC4の内部状態を完全に解析しなくても正解を導出できた。

### 教訓
- RC4単体では既知平文に弱い
- ストリーム暗号のキーストリームは再利用してはならない

## 補足 x32dbg
### Riddle.exeをアタッチしてトレース
1. Riddle.exeを起動し、3問目まで頑張って解く
1. [ファイル]-[アタッチ]でRiddleを選択する
1. [CPU]タブ内で右クリックし、[検索]-[すべてのユーザーモジュール]-[文字列参照]を選択
1. "Correct！"をダブルクリックし、表示されたアセンブリを遡り、確認
1. アドレス006B19E5でサブルーチンをコールしている（call <riddle.sub_6B17D0>）
1. アドレス006B1A02で暗号化済み入力（ESI)と 正解暗号文（ECX）を比較している（cmp edx,dword ptr ds:[ecx]）
1. 次のステップで比較結果が一致しなければ、"Wrong!"の表示へジャンプ（jne riddle.6B1A7D）

アドレス006B19E5のcall <riddle.sub_6B17D0>の行をダブルクリックし、サブルーチンのアセンブリを確認
- アドレス006B1963のxor byte ptr ds:[eax+esi],clで、XORしてキーストリームを算出
- アドレス0x6B5578 はRC4のSボックス（256バイトの配列）
- xor byte ptr ds:[eax+esi], cl の cl が固定値ではなく毎回変化するキーストリーム。

|アドレス|バイト列|ASCII|
|---|:---|:---|
| 006B5578 | 38 49 1B 3B E6 63 DC 66 12 28 08 42 7C C2 BB 24 | 8I.;æcÜf.(.B|Â»$  |
| 006B5588 | C5 1C 89 2E 20 D9 5E B1 78 65 D6 55 27 1D 67 52 | Å... Ù^±xeÖU'.gR  |
| 006B5598 | 7B 33 76 04 06 A2 5A 83 82 0F 00 62 D3 30 AF 2F | {3v..¢Z....bÓ0¯/  |
| 006B55A8 | A3 CD A1 9B AE 6D EE 69 E8 91 F7 E0 D4 2D EF 03 | £Í¡.®mîiè.÷àÔ-ï.  |
| 006B55B8 | ED 81 2B 21 56 85 45 FF 8E 84 BC 8B 7A 98 37 10 | í.+!V.Eÿ..¼.z.7.  |
| 006B55C8 | FD 68 01 11 F4 C8 22 BD 75 FB B7 23 DA 15 14 A0 | ýh..ôÈ"½uû·#Ú..   |
| 006B55D8 | E4 32 A5 0B C9 26 F9 AC D1 97 BE DE 8F 51 02 29 | ä2¥.É&ù¬Ñ.¾Þ.Q.)  |
| 006B55E8 | 79 0C 71 CF 5F 80 86 64 4C 13 F1 F0 90 1F 40 57 | y.qÏ_..dL.ñð..@W  |
| 006B55F8 | D2 77 61 F8 D7 C7 B3 7E 6B 31 EC 1A 09 72 47 B4 | Òwaø×Ç³~k1ì..rG´  |
| 006B5608 | CA 95 C1 99 A7 5B 94 C0 6C 9C 54 B8 D8 5D 1E E7 | Ê.Á.§[.Àl.T¸Ø].ç  |
| 006B5618 | 3C 60 7D BA 39 AD 07 34 93 DB 9F CE 88 B6 3A B9 | <`}º9..4.Û.Î.¶:¹  |
| 006B5628 | 6E 58 59 B2 CC 0E 35 F5 D5 E9 CB DF BF 46 E1 8D | nXY²Ì.5õÕéËß¿Fá.  |
| 006B5638 | A6 4A 0A EB DD B0 AA 74 9E 7F 4F A4 E3 FA 6F 9A | ¦J.ëÝ°ªt..O¤ãúo.  |
| 006B5648 | 3F 3D E2 F3 C6 C4 6A 73 8C 48 9D E5 B5 4B 4E 18 | ?=âóÆÄjs.H.åµKN.  |
| 006B5658 | 53 3E 87 92 05 5C F6 44 0D EA A8 43 A9 8A FE D0 | S>...\öD.ê¨C©.þÐ  |
| 006B5668 | 41 50 AB FC F2 19 36 70 25 2C 4D 16 C3 17 2A 96 | AP«üò.6p%,M.Ã.*.  |

| アドレス | バイト列 | ニーモニック | コメント |
|---|:---|:---|:---|
| 006B17D0  | 55                          | push ebp                                     |
| 006B17D1  | 8BEC                        | mov ebp,esp                                  |
| 006B17D3  | 83EC 0C                     | sub esp,C                                    |
| 006B17D6  | 33C0                        | xor eax,eax                                  | eax:"FLAG_SRORGLnTh2Q5fTwu"
| 006B17D8  | EB 06                       | jmp riddle.6B17E0                            |
| 006B17DA  | 8D9B 00000000               | lea ebx,dword ptr ds:[ebx]                   | ebx:"\\8k"
| 006B17E0  | 8880 78556B00               | mov byte ptr ds:[eax+6B5578],al              |
| 006B17E6  | 40                          | inc eax                                      | eax:"FLAG_SRORGLnTh2Q5fTwu"
| 006B17E7  | 3D 00010000                 | cmp eax,100                                  | eax:"FLAG_SRORGLnTh2Q5fTwu"
| 006B17EC  | 7C F2                       | jl riddle.6B17E0                             |
| 006B17EE  | 53                          | push ebx                                     | ebx:"\\8k"
| 006B17EF  | BA 02000000                 | mov edx,2                                    |
| 006B17F4  | 81EA 78556B00               | sub edx,riddle.6B5578                        |
| 006B17FA  | 56                          | push esi                                     |
| 006B17FB  | 57                          | push edi                                     |
| 006B17FC  | 8955 F8                     | mov dword ptr ss:[ebp-8],edx                 |
| 006B17FF  | BF 01000000                 | mov edi,1                                    |
| 006B1804  | BA 03000000                 | mov edx,3                                    |
| 006B1809  | 32C9                        | xor cl,cl                                    |
| 006B180B  | 33C0                        | xor eax,eax                                  | eax:"FLAG_SRORGLnTh2Q5fTwu"
| 006B180D  | 81EF 78556B00               | sub edi,riddle.6B5578                        |
| 006B1813  | 81EA 78556B00               | sub edx,riddle.6B5578                        |
| 006B1819  | 8955 F4                     | mov dword ptr ss:[ebp-C],edx                 |
| 006B181C  | 8D6424 00                   | lea esp,dword ptr ss:[esp]                   | [esp]:"FLAG_SRORGLnTh2Q5fTwu"
| 006B1820  | 8A90 78556B00               | mov dl,byte ptr ds:[eax+6B5578]              |
| 006B1826  | 8BF0                        | mov esi,eax                                  | eax:"FLAG_SRORGLnTh2Q5fTwu"
| 006B1828  | 81E6 0F000080               | and esi,8000000F                             |
| 006B182E  | 79 05                       | jns riddle.6B1835                            |
| 006B1830  | 4E                          | dec esi                                      |
| 006B1831  | 83CE F0                     | or esi,FFFFFFF0                              |
| 006B1834  | 46                          | inc esi                                      |
| 006B1835  | 0FB69E 18386B00             | movzx ebx,byte ptr ds:[esi+6B3818]           | ebx:"\\8k"
| 006B183C  | 02DA                        | add bl,dl                                    |
| 006B183E  | 02CB                        | add cl,bl                                    |
| 006B1840  | 0FB6F1                      | movzx esi,cl                                 |
| 006B1843  | 8A9E 78556B00               | mov bl,byte ptr ds:[esi+6B5578]              |
| 006B1849  | 8896 78556B00               | mov byte ptr ds:[esi+6B5578],dl              |
| 006B184F  | 8A90 79556B00               | mov dl,byte ptr ds:[eax+6B5579]              |
| 006B1855  | 8DB407 78556B00             | lea esi,dword ptr ds:[edi+eax+6B5578]        |
| 006B185C  | 81E6 0F000080               | and esi,8000000F                             |
| 006B1862  | 8898 78556B00               | mov byte ptr ds:[eax+6B5578],bl              |
| 006B1868  | 79 05                       | jns riddle.6B186F                            |
| 006B186A  | 4E                          | dec esi                                      |
| 006B186B  | 83CE F0                     | or esi,FFFFFFF0                              |
| 006B186E  | 46                          | inc esi                                      |
| 006B186F  | 0FB69E 18386B00             | movzx ebx,byte ptr ds:[esi+6B3818]           | ebx:"\\8k"
| 006B1876  | 02DA                        | add bl,dl                                    |
| 006B1878  | 02CB                        | add cl,bl                                    |
| 006B187A  | 0FB6F1                      | movzx esi,cl                                 |
| 006B187D  | 8A9E 78556B00               | mov bl,byte ptr ds:[esi+6B5578]              |
| 006B1883  | 8896 78556B00               | mov byte ptr ds:[esi+6B5578],dl              |
| 006B1889  | 8B75 F8                     | mov esi,dword ptr ss:[ebp-8]                 |
| 006B188C  | 8A90 7A556B00               | mov dl,byte ptr ds:[eax+6B557A]              |
| 006B1892  | 8DB406 78556B00             | lea esi,dword ptr ds:[esi+eax+6B5578]        |
| 006B1899  | 81E6 0F000080               | and esi,8000000F                             |
| 006B189F  | 8898 79556B00               | mov byte ptr ds:[eax+6B5579],bl              |
| 006B18A5  | 79 05                       | jns riddle.6B18AC                            |
| 006B18A7  | 4E                          | dec esi                                      |
| 006B18A8  | 83CE F0                     | or esi,FFFFFFF0                              |
| 006B18AB  | 46                          | inc esi                                      |
| 006B18AC  | 0FB69E 18386B00             | movzx ebx,byte ptr ds:[esi+6B3818]           | ebx:"\\8k"
| 006B18B3  | 02DA                        | add bl,dl                                    |
| 006B18B5  | 02CB                        | add cl,bl                                    |
| 006B18B7  | 0FB6F1                      | movzx esi,cl                                 |
| 006B18BA  | 8A9E 78556B00               | mov bl,byte ptr ds:[esi+6B5578]              |
| 006B18C0  | 8896 78556B00               | mov byte ptr ds:[esi+6B5578],dl              |
| 006B18C6  | 8B75 F4                     | mov esi,dword ptr ss:[ebp-C]                 |
| 006B18C9  | 8A90 7B556B00               | mov dl,byte ptr ds:[eax+6B557B]              |
| 006B18CF  | 8DB406 78556B00             | lea esi,dword ptr ds:[esi+eax+6B5578]        |
| 006B18D6  | 81E6 0F000080               | and esi,8000000F                             |
| 006B18DC  | 8898 7A556B00               | mov byte ptr ds:[eax+6B557A],bl              |
| 006B18E2  | 79 05                       | jns riddle.6B18E9                            |
| 006B18E4  | 4E                          | dec esi                                      |
| 006B18E5  | 83CE F0                     | or esi,FFFFFFF0                              |
| 006B18E8  | 46                          | inc esi                                      |
| 006B18E9  | 0FB69E 18386B00             | movzx ebx,byte ptr ds:[esi+6B3818]           | ebx:"\\8k"
| 006B18F0  | 02DA                        | add bl,dl                                    |
| 006B18F2  | 02CB                        | add cl,bl                                    |
| 006B18F4  | 0FB6F1                      | movzx esi,cl                                 |
| 006B18F7  | 8A9E 78556B00               | mov bl,byte ptr ds:[esi+6B5578]              |
| 006B18FD  | 8896 78556B00               | mov byte ptr ds:[esi+6B5578],dl              |
| 006B1903  | 8898 7B556B00               | mov byte ptr ds:[eax+6B557B],bl              |
| 006B1909  | 83C0 04                     | add eax,4                                    | eax:"FLAG_SRORGLnTh2Q5fTwu"
| 006B190C  | 3D 00010000                 | cmp eax,100                                  | eax:"FLAG_SRORGLnTh2Q5fTwu"
| 006B1911  | 0F8C 09FFFFFF               | jl riddle.6B1820                             |
| 006B1917  | 33C0                        | xor eax,eax                                  | eax:"FLAG_SRORGLnTh2Q5fTwu"
| 006B1919  | 32D2                        | xor dl,dl                                    |
| 006B191B  | 32DB                        | xor bl,bl                                    |
| 006B191D  | 3945 0C                     | cmp dword ptr ss:[ebp+C],eax                 |
| 006B1920  | 7E 4A                       | jle riddle.6B196C                            |
| 006B1922  | EB 03                       | jmp riddle.6B1927                            |
| 006B1924  | 8A5D FF                     | mov bl,byte ptr ss:[ebp-1]                   |
| 006B1927  | FEC2                        | inc dl                                       |
| 006B1929  | 0FB6F2                      | movzx esi,dl                                 |
| 006B192C  | 8A8E 78556B00               | mov cl,byte ptr ds:[esi+6B5578]              |
| 006B1932  | 02D9                        | add bl,cl                                    |
| 006B1934  | 0FB6FB                      | movzx edi,bl                                 |
| 006B1937  | 885D FF                     | mov byte ptr ss:[ebp-1],bl                   |
| 006B193A  | 0FB69F 78556B00             | movzx ebx,byte ptr ds:[edi+6B5578]           | ebx:"\\8k"
| 006B1941  | 889E 78556B00               | mov byte ptr ds:[esi+6B5578],bl              |
| 006B1947  | 888F 78556B00               | mov byte ptr ds:[edi+6B5578],cl              |
| 006B194D  | 0FB69E 78556B00             | movzx ebx,byte ptr ds:[esi+6B5578]           | ebx:"\\8k"
| 006B1954  | 8B75 08                     | mov esi,dword ptr ss:[ebp+8]                 | [ebp+08]:"\\8k"
| 006B1957  | 02D9                        | add bl,cl                                    |
| 006B1959  | 0FB6CB                      | movzx ecx,bl                                 | ecx:Ordinal#6969+73
| 006B195C  | 0FB689 78556B00             | movzx ecx,byte ptr ds:[ecx+6B5578]           | ecx:Ordinal#6969+73
| 006B1963  | 300C30                      | xor byte ptr ds:[eax+esi],cl                 |
| 006B1966  | 40                          | inc eax                                      | eax:"FLAG_SRORGLnTh2Q5fTwu"
| 006B1967  | 3B45 0C                     | cmp eax,dword ptr ss:[ebp+C]                 |
| 006B196A  | 7C B8                       | jl riddle.6B1924                             |
| 006B196C  | 5F                          | pop edi                                      |
| 006B196D  | 5E                          | pop esi                                      |
| 006B196E  | 5B                          | pop ebx                                      | ebx:"\\8k"
| 006B196F  | 8BE5                        | mov esp,ebp                                  |
| 006B1971  | 5D                          | pop ebp                                      |
| 006B1972  | C3                          | ret                                          |

| 006B1A02  | 3B11                        | cmp edx,dword ptr ds:[ecx]                   |でレジスタの値を確認
- ECX:006B51F4のダンプを確認 … 正解暗号文
- ESX:0117ED18のダンプを確認 … 暗号化済み入力

|レジスタ|値|
|:---|:---|
|EAX | 00000015|
|EBX | 0117F568     "\\8k"|
|ECX | 006B51F4     riddle.006B51F4|
|EDX | 0D1BFFBF|
|EBP | 0117ED3C|
|ESP | 0117ED0C|
|ESI | 0117ED18|
|EDI | 00000111     L'đ'|
|EIP | 006B1A02     riddle.006B1A02|
|EFLAGS | 00000304     L'̄'|
|ZF | 0|
|OF | 0|
|CF | 0     L'Ā'|
|PF | 1|
|SF | 0|
|TF | 1     L'ā'|
|AF | 0|
|DF | 0|
|IF | 1|
|LastError | 00000000 (ERROR_SUCCESS)|
|LastStatus | C0000034 (STATUS_OBJECT_NAME_NOT_FOUND)|
|GS | 002B|
|ES | 002B|
|CS | 0023|
|FS | 0053|
|DS | 002B|
|SS | 002B|
|ST(0) | 400B8C28000000000000|
|ST(1) | 400AAE01D3B597796000|
|ST(2) | 40079AAC4A6886A4C800|
|ST(3) | 400BE1C8000000000000|
|ST(4) | 3FFF8000000000000000|
|ST(5) | 3FFF8000000000000000|
|ST(6) | 40038000000000000000|
|ST(7) | 40038000000000000000|
|x87TagWord | FFFF|
|x87ControlWord | 027F|
|x87StatusWord | 4020|
|x87TW_0 | 3 (空)|
|x87TW_1 | 3 (空)|
|x87TW_2 | 3 (空)|
|x87TW_3 | 3 (空)|
|x87TW_4 | 3 (空)|
|x87TW_5 | 3 (空)|
|x87TW_6 | 3 (空)|
|x87TW_7 | 3 (空)|
|x87SW_B | 0     L'Ā'|
|x87SW_C3 | 1|
|x87SW_TOP | 0 (ST0=x87r0)|
|x87SW_C2 | 0|
|x87SW_C1 | 0|
|x87SW_O | 0|
|x87SW_ES | 0|
|x87SW_SF | 0     L'Ā'|
|x87SW_P | 1|
|x87SW_U | 0|
|x87SW_Z | 0|
|x87SW_D | 0|
|x87SW_I | 0|
|x87SW_C0 | 0|
|x87CW_IC | 0|
|x87CW_RC | 0 (最近接丸め)|
|x87CW_PC | 2 (Real8)|
|x87CW_PM | 1|
|x87CW_UM | 1|
|x87CW_OM | 1|
|x87CW_ZM | 1|
|x87CW_DM | 1     L'ā'|
|x87CW_IM | 1|
|MxCsr | 00001FA0|
|MxCsr_FZ | 0|
|MxCsr_PM | 1|
|MxCsr_UM | 1|
|MxCsr_OM | 1|
|MxCsr_ZM | 1|
|MxCsr_IM | 1|
|MxCsr_DM | 1|
|MxCsr_DAZ | 0     L'Ā'|
|MxCsr_PE | 1|
|MxCsr_UE | 0|
|MxCsr_OE | 0|
|MxCsr_ZE | 0|
|MxCsr_DE | 0|
|MxCsr_IE | 0|
|MxCsr_RC | 0 (最近接丸め)|
|XMM0  | 00000000000000000000000000000000|
|XMM1  | 00000000000000000000000000000000|
|XMM2  | 00000000000000000000000000000000|
|XMM3  | 00000000000000000000000000000000|
|XMM4  | 00000000000000000000000000000000|
|XMM5  | 00000000000000000000000000000000|
|XMM6  | 00000000000000000000000000000000|
|XMM7  | 00010000000100000000000000000000|
|K0  | 0000000000000000|
|K1  | 0000000000000000|
|K2  | 0000000000000000|
|K3  | 0000000000000000|
|K4  | 0000000000000000|
|K5  | 0000000000000000|
|K6  | 0000000000000000|
|K7  | 0000000000000000|
|DR0 | 00000000|
|DR1 | 00000000|
|DR2 | 00000000|
|DR3 | 00000000|
|DR6 | 00000000|
|DR7 | 00000000|

|アドレス|バイト列|ASCII|
|---|:---|:---|
|006B51F4 | BF FF 1B 0D 47 A7 18 4F CB D6 5C 59 95 61 73 57 | ¿ÿ..G§.OËÖ\Y.asW  |
|006B5204 | B3 D1 94 9F AC 00 00 00 00 00 00 00 00 00 00 00 | ³Ñ..¬...........  |

|アドレス|バイト列|ASCII|
|---|:---|:---|
|0117ED18 | BF FF 1B 0D 47 B9 0F 79 CB A5 4A 51 F5 48 75 4E | ¿ÿ..G¹.yË¥JQõHuN  |
|0117ED28 | C0 E0 F7 DB B2 00 17 01 C3 EF 17 01 C8 F1 17 01 | Àà÷Û²...Ãï..Èñ..  |
